In [5]:
# day30_performance_pipeline.py

import time
import asyncio
from typing import List, Dict
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from concurrent.futures import ThreadPoolExecutor

# Mock functions (replace with real OpenAI + FAISS)
def embed(text: str) -> List[float]:
    time.sleep(0.2);
    return [0.1] * 768

def faiss_search(vector: List[float]) -> List[str]:
    time.sleep(0.1);
    return ["doc1", "doc2", "doc3"]

def build_prompt(query: str, docs: List[str]) -> str:
    return f"Answer using: {docs}\nQuestion: {query}"

async def llm_generate_stream(prompt: str):
    # simulate streaming tokens
    tokens = ["This ", "is ", "a ", "streamed ", "response."]
    for t in tokens:
        await asyncio.sleep(0.3);
        yield t

# ----------------------------
# Timing Utility
# ----------------------------
def timed_step(name: str, func, *args, **kwargs):
    start = time.perf_counter()
    result = func(*args, **kwargs)
    end = time.perf_counter()
    return result, end - start

# ----------------------------
# Async Embedding (Parallel)
# ----------------------------
async def embed_async(text: str):
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(None, embed, text)

async def embed_all(texts: List[str]):
    tasks = [embed_async(t) for t in texts]
    return await asyncio.gather(*tasks)

# ----------------------------
# FastAPI App
# ----------------------------
app = FastAPI()

@app.get("/query")
async def query(q: str):

    timings: Dict[str, float] = {}

    # 1. Embedding (Async Parallel)
    start = time.perf_counter()
    embeddings = await embed_all([q])
    timings["embedding"] = time.perf_counter() - start

    # 2. FAISS Search
    docs, t = timed_step("faiss", faiss_search, embeddings[0])
    timings["faiss"] = t

    # 3. Prompt Construction
    prompt, t = timed_step("prompt", build_prompt, q, docs)
    timings["prompt"] = t

    # 4. Streaming Response
    async def stream():
        start_total = time.perf_counter()

        # measure TTFT
        first_token_time = None

        async for token in llm_generate_stream(prompt):
            if first_token_time is None:
                first_token_time = time.perf_counter()

            yield token

        end_total = time.perf_counter()

        # Log streaming metrics
        print("\n--- STREAM METRICS ---")
        print(f"TTFT: {first_token_time - start_total:.2f}s")
        print(f"Full Response: {end_total - start_total:.2f}s")

        print("\n--- STEP TIMINGS ---")
        total = sum(timings.values())
        for k, v in timings.items():
            print(f"{k}: {v:.2f}s ({(v/total)*100:.1f}%)")

    return StreamingResponse(stream(), media_type="text/plain")


# ----------------------------
# Benchmark Runner (20 Queries)
# ----------------------------
async def benchmark():

    queries = [f"query {i}" for i in range(20)]

    total_times = {
        "embedding": 0,
        "faiss": 0,
        "prompt": 0,
        "llm": 0
    }

    for q in queries:

        # Embedding
        start = time.perf_counter()
        emb = await embed_all([q])
        total_times["embedding"] += time.perf_counter() - start

        # FAISS
        _, t = timed_step("faiss", faiss_search, emb[0])
        total_times["faiss"] += t

        # Prompt
        prompt, t = timed_step("prompt", build_prompt, q, ["doc"])
        total_times["prompt"] += t

        # LLM (simulate)
        start = time.perf_counter()
        async for _ in llm_generate_stream(prompt):
            pass
        total_times["llm"] += time.perf_counter() - start

    print("\n=== AVERAGE TIMINGS (20 QUERIES) ===")
    total = sum(total_times.values())

    for k, v in total_times.items():
        avg = v / 20
        print(f"{k}: {avg:.2f}s ({(v/total)*100:.1f}%)")


# ----------------------------
# Run Benchmark (manual)
# ----------------------------
# if __name__ == "__main__": # This block is usually not needed in Colab for direct async execution
await benchmark()



=== AVERAGE TIMINGS (20 QUERIES) ===
embedding: 0.20s (11.1%)
faiss: 0.10s (5.6%)
prompt: 0.00s (0.0%)
llm: 1.50s (83.3%)
